In [1]:
import os
from langchain.agents import create_agent
from dotenv import load_dotenv

c:\Delta Training Local\gen-ai\Langchain agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your environment or .env file.")
os.environ["GROQ_API_KEY"] = api_key
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_BASE_URL"] = os.getenv("LANGFUSE_BASE_URL")

In [17]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [18]:
from langchain.tools import tool

@tool("add_numbers")
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool("subtract_numbers")
def subtract_numbers(a: int, b: int) -> int:
    """Subtract the second number from the first."""
    return a - b
@tool("multiply_numbers")
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b
@tool("divide_numbers")
def divide_numbers(a: int, b: int) -> float:
    """Divide the first number by the second."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

In [19]:
tools = [add_numbers, subtract_numbers, multiply_numbers, divide_numbers]
agent=create_agent(llm,
    tools,
    system_prompt="""You are a helpful assistant that can perform basic arithmetic operations. 
When given a question, determine which tool to use and call it with the appropriate arguments.
Only use the tools provided, and do not attempt to perform calculations yourself.""")

In [ ]:
from langfuse.langchain import CallbackHandler
from langfuse import get_client
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [27]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "What is 5 plus 3 multiplied by 5 and added with 5?"}]
},config={"callbacks": [langfuse_handler]})
result["messages"][-1].content

'The final result of the calculation is **25**.'